# Applio Voice Model Training on Google Colab

This notebook prepares the official Applio environment, mounts Google Drive, copies the vocal dataset into Applio's expected dataset directory, and launches Applio. Training is then completed in Applio's Train tab.

**Use only voice recordings for which you have permission.** Keep the Colab runtime on GPU and save checkpoints to Drive because Colab sessions can disconnect.

In [ ]:
# 1) Check the Colab runtime GPU
# Colab UI reports the selected GPU; some current runtimes do not expose nvidia-smi.
import os, shutil, subprocess
nvidia = shutil.which('nvidia-smi')
if nvidia:
    subprocess.run([nvidia], check=False)
else:
    print('nvidia-smi is unavailable in this Colab runtime; continuing because GPU runtime is selected in the UI.')
print('GPU check completed.')

In [ ]:
# 2) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/Applio_Training'
SOURCE_DATASET = os.path.join(DRIVE_ROOT, 'dataset')
OUTPUT_MODELS = os.path.join(DRIVE_ROOT, 'output_models')
os.makedirs(OUTPUT_MODELS, exist_ok=True)
assert os.path.isdir(SOURCE_DATASET), f'Dataset folder not found: {SOURCE_DATASET}'
print('Dataset:', SOURCE_DATASET)
print('Model output:', OUTPUT_MODELS)

In [ ]:
# 3) Clone the official Applio repository
%cd /content
!rm -rf Applio
!git clone --depth 1 https://github.com/iahispano/Applio.git
%cd /content/Applio
!bash run-install.sh

In [ ]:
# 4) Copy and normalize the Drive dataset
# Applio expects a model-specific folder under assets/datasets.
import shutil, subprocess, pathlib
MODEL_NAME = 'Custom_Voice_Model'
APPLIO_DATASET = f'/content/Applio/assets/datasets/{MODEL_NAME}'
shutil.rmtree(APPLIO_DATASET, ignore_errors=True)
os.makedirs(APPLIO_DATASET, exist_ok=True)

source_files = []
for path in pathlib.Path(SOURCE_DATASET).rglob('*'):
    if path.is_file() and path.suffix.lower() in {'.wav', '.flac', '.m4a', '.mp3', '.ogg'}:
        source_files.append(path)
assert source_files, f'No audio files found in {SOURCE_DATASET}'

converted = 0
for source in sorted(source_files):
    target = pathlib.Path(APPLIO_DATASET) / (source.stem + '.wav')
    if source.suffix.lower() == '.wav':
        shutil.copy2(source, target)
    else:
        subprocess.run([
            'ffmpeg', '-y', '-i', str(source), '-ac', '1', '-ar', '40000',
            '-sample_fmt', 's16', str(target)
        ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        converted += 1

audio_files = sorted(pathlib.Path(APPLIO_DATASET).glob('*.wav'))
print(f'Prepared {len(audio_files)} WAV files in {APPLIO_DATASET}')
print(f'Converted compressed files: {converted}')
for path in audio_files:
    print(f' - {path.name}: {path.stat().st_size / 1024 / 1024:.1f} MB')

In [ ]:
# 5) Basic audio duration check
import wave
total_seconds = 0.0
for path in audio_files:
    try:
        with wave.open(str(path), 'rb') as audio:
            duration = audio.getnframes() / audio.getframerate()
            total_seconds += duration
            print(f'{path.name}: {duration / 60:.2f} min, {audio.getframerate()} Hz, {audio.getnchannels()} channel(s)')
    except wave.Error as error:
        raise RuntimeError(f'Unreadable WAV file: {path}: {error}')
print(f'Total readable WAV duration: {total_seconds / 60:.2f} minutes')
if total_seconds < 10 * 60:
    print('WARNING: official guidance recommends approximately 10-30 minutes of clean audio.')

## 6) Launch Applio

Run the next cell. When the public Gradio URL appears, open it. In the **Train** tab use:

- Model name: `Custom_Voice_Model`
- Dataset folder: the prepared model folder under `assets/datasets`
- Sample rate: choose `40k` or `48k` consistently with the selected pre-trained model
- Pitch extraction: `RMVPE`
- Save every epoch: `25`
- Total epochs: start around `300`, then monitor TensorBoard/loss and stop or resume based on quality
- Batch size: start conservatively; halve it if CUDA out-of-memory occurs

After training, use **Train Index**, then **Export Model**. Export the matching `.pth` and `.index` files into the Drive `output_models` folder.

In [ ]:
# 7) Launch the official Applio UI
%cd /content/Applio
!python app.py --share

In [ ]:
# 8) Optional: verify exported model files after training
import glob, os
pth_files = glob.glob(os.path.join(OUTPUT_MODELS, '*.pth'))
index_files = glob.glob(os.path.join(OUTPUT_MODELS, '*.index'))
print('PTH files:')
for path in pth_files: print(' -', os.path.basename(path), os.path.getsize(path))
print('Index files:')
for path in index_files: print(' -', os.path.basename(path), os.path.getsize(path))
assert pth_files and index_files, 'Export both a .pth and a matching .index file before finishing.'